In [50]:
import os
import sys
from glob import glob
from collections import namedtuple
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as clr
import cmocean.cm as cmo


# --- I/O utilities -----------------------------------------------------------

def _shift_time(da):
    """
    Correct time coordinate when CLM5 outputs month-end timestamps.
    Detects the case where the first timestamp is February and the last is
    January (i.e., shifted by one month), and reassigns the coordinate to
    calendar-aligned month-start values.
    """
    if (da.time[0].dt.month.item() == 2) and (da.time[-1].dt.month.item() == 1):
        new_time = xr.date_range(
            start=str(da.time[0].dt.year.item()) + "-01",
            end=str(da.time[-1].dt.year.item() - 1) + "-12",
            freq="MS",
            calendar="noleap",
            use_cftime=True,
        )
        return da.assign_coords(time=new_time)
    return da


def check_frequency(ds):
    """
    Infer temporal frequency of a dataset from the number of time steps per year.
    Returns 'monthly', 'yearly', or 'unknown'.
    """
    time_steps_per_year = len(ds.time) / (ds.time[-1].dt.year - ds.time[0].dt.year + 1)
    if time_steps_per_year == 12:
        freq = "monthly"
    elif time_steps_per_year == 1:
        freq = "yearly"
    else:
        freq = "unknown"
    return freq


def load_variables(
    varnames,
    case,
    basedir,
    domain="lnd",
    htape="h0",
    suffix="",
    chunks={'time': -1},
    parallel=True,
):
    """
    Load one or more variables from CESM2 history files using xarray.open_mfdataset.

    Parameters
    ----------
    varnames : list of str
        Variable names to load. If 'PRECT' is included, it is computed as
        PRECC + PRECL (convective + large-scale precipitation rates).
    case : str
        CESM2 case name used to construct the file path pattern.
    basedir : str
        Base archive directory (e.g., GLADE scratch or campaign storage).
    domain : {'lnd', 'atm'}
        Model component domain; determines component string in filename.
    htape : str
        History tape identifier (e.g., 'h0', 'h1').
    suffix : str
        Optional suffix appended to the case directory name.
    chunks : dict or None
        Dask chunking passed to open_mfdataset. If None, a time-forward chunk
        layout is used.
    parallel : bool
        Whether to enable parallel file reads in open_mfdataset.

    Returns
    -------
    xr.Dataset
        Dataset with time coordinate shifted if necessary (see _shift_time).
    """

    def _keep_var(ds):
        if "PRECT" in varnames:
            x = ds["PRECC"] + ds["PRECL"]
            x = x.rename("PRECT").assign_attrs(
                units="m/s",
                long_name="calculated total precipitation rate (liq + ice)",
            )
            other_varnames = [v for v in varnames if v != "PRECT"]
            if other_varnames:
                return xr.merge([ds[other_varnames], x])
            return x.to_dataset()
        return ds[varnames]

    component = {
        "lnd": "clm2",
        "atm": "cam",
    }

    if len(suffix):
        suffix = "." + suffix

    if chunks is None:
        chunks = {"time": 120}

    preprocess = _keep_var if varnames else None

    data = xr.open_mfdataset(
        f"{basedir}/{case}{suffix}/{domain}/hist/{case}.{component[domain]}.{htape}.*.nc",
        combine="by_coords",
        decode_timedelta=False,
        preprocess=preprocess,
        engine="netcdf4",
        chunks=chunks,
        parallel=parallel,
    )

    return _shift_time(data)


# --- Diagnostics plotting ----------------------------------------------------

def plot_simple_diags(ihx, ih0, fhx, variables, tag, to_save=True):
    """
    Produce a four-panel diagnostic figure for each variable:
      col 0: global annual time series (land-area weighted sum or mean)
      col 1: climatological seasonal cycle over CLIM_YEAR_RANGE
      col 2: spatial map of ih0 climatology over MAP_YEAR_RANGE
      col 3: spatial map of (ihx - ih0) difference over MAP_YEAR_RANGE

    Comparisons are between:
      ihx  : interactive run, perturbed member (CASE)
      ih0  : interactive run, control member (coupPPE.000)
      fhx  : offline (FHIST) run, perturbed member (CASE)

    Parameters
    ----------
    ihx, ih0, fhx : xr.Dataset
        Datasets for the three simulations, each containing the variables
        listed in `variables`.
    variables : list of str
        Variable names to plot; must be keys in the module-level `cfs` dict.
    tag : str
        String appended to the output filename (e.g., 'lnd' or 'atm').
    to_save : bool
        If True, save figure to disk and close; otherwise display inline.
    """
    print("variables:", variables)

    fig, axes = plt.subplots(
        nrows=len(variables),
        ncols=4,
        figsize=(20, 4 * len(variables)),
        layout="tight",
    )
    ax = axes.flatten()

    for i, v in enumerate(variables):
        print(v)

        cm = cmaps["temp"]
        if v in cmaps["veg"]["vars"]:
            cm = cmaps["veg"]
        elif v in cmaps["water"]["vars"]:
            cm = cmaps["water"]
        elif v in cmaps["temp"]["vars"]:
            cm = cmaps["temp"]

        vmin = 0
        if v in ["TSA", "TREFHT"]:
            vmin = None

        transformed = {
            "ihx": (ihx[v] + cfs[v].cs) * cfs[v].cf,
            "ih0": (ih0[v] + cfs[v].cs) * cfs[v].cf,
            "fhx": (fhx[v] + cfs[v].cs) * cfs[v].cf,
        }

        annual = xr.Dataset(
            {
                name: da.sum(dim=["lat", "lon"]).groupby("time.year").mean()
                for name, da in transformed.items()
            }
        ).persist()

        smooth_window = 5
        annual_smooth = annual.rolling(year=smooth_window, center=True, min_periods=1).mean().persist()

        # Materialize once per variable, rather than repeatedly inside each .plot call.
        annual = annual.compute()
        annual_smooth = annual_smooth.compute()

        annual["ihx"].plot(ax=ax[4 * i], color="tab:blue", alpha=0.75, lw=0.75)
        annual_smooth["ihx"].plot(
            ax=ax[4 * i], color="tab:blue", alpha=1, lw=1, label=f"I {CASE}"
        )
        annual["ih0"].plot(ax=ax[4 * i], color="tab:orange", alpha=0.75, lw=0.75)
        annual_smooth["ih0"].plot(
            ax=ax[4 * i], color="tab:orange", alpha=1, lw=1, label="I coupPPE.000"
        )
        annual["fhx"].plot(ax=ax[4 * i], color="tab:red", alpha=0.75, lw=0.75)
        annual_smooth["fhx"].plot(
            ax=ax[4 * i], color="tab:green", alpha=1, lw=1, label=f"F {CASE}"
        )

        ax[4 * i].set_ylabel(f"{v} [{cfs[v].unit}]")
        ax[4 * i].set_title(f"global annual {labels[cfs[v].kind]} {v}")
        ax[4 * i].legend()

        clim = xr.Dataset(
            {
                name: da.sel(time=slice(CLIM_YEAR_RANGE[0], CLIM_YEAR_RANGE[1]))
                .sum(dim=["lat", "lon"])
                .groupby("time.month")
                .mean()
                for name, da in transformed.items()
            }
        ).compute()

        clim["ihx"].plot(ax=ax[4 * i + 1], color="tab:blue", label=f"I {CASE}")
        clim["ih0"].plot(ax=ax[4 * i + 1], color="tab:orange", label="I coupPPE.000")
        clim["fhx"].plot(ax=ax[4 * i + 1], color="tab:red", label=f"F {CASE}")

        ax[4 * i + 1].set_ylabel(f"{v} [{cfs[v].unit}]")
        ax[4 * i + 1].set_xlabel("month")
        ax[4 * i + 1].set_title(
            f"global clim {labels[cfs[v].kind]} {v} "
            f"{CLIM_YEAR_RANGE[0][:4]}-{CLIM_YEAR_RANGE[1][:4]}"
        )
        ax[4 * i + 1].legend()

        base_map = (ih0[v] + cfs[v].cs).sel(time=slice(MAP_YEAR_RANGE[0], MAP_YEAR_RANGE[1])).mean(dim="time")
        diff_map = (
            (ihx[v] + cfs[v].cs).sel(time=slice(MAP_YEAR_RANGE[0], MAP_YEAR_RANGE[1]))
            - (ih0[v] + cfs[v].cs).sel(time=slice(MAP_YEAR_RANGE[0], MAP_YEAR_RANGE[1]))
        ).mean(dim="time")
        base_map = base_map.compute()
        diff_map = diff_map.compute()

        base_map.plot(
            ax=ax[4 * i + 2],
            vmin=vmin,
            cmap=cm["cont"],
            cbar_kwargs={"label": f"{v} [{cfs[v].unit}]"},
        )
        ax[4 * i + 2].set_title(f"{CASE} {v} {MAP_YEAR_RANGE[0][:4]}")

        diff_map.plot(
            ax=ax[4 * i + 3],
            cmap=cm["diff"],
            norm=clr.CenteredNorm(),
            robust=True,
            cbar_kwargs={"label": f"{v} [{cfs[v].unit}]"},
        )
        ax[4 * i + 3].set_title(f"{CASE[-3:]}$-$000 {v} {MAP_YEAR_RANGE[0][:4]}")

        for j in range(2, 4):
            ax[4 * i + j].set_ylabel("")
            ax[4 * i + j].set_xlabel("")

    if to_save:
        fig.savefig(
            f"{SIM_DIR}/{CASE}/{CASE_PREFIX}.{CASE}.{tag}.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.close()

In [20]:
# --- Configuration -----------------------------------------------------------

CASE        = "coupPPE.001"
CASE_PREFIX = "i.e21.CPLHIST_BGC.f19_f19_mg17.historical"
SIM_DIR     = "/glade/u/home/bbuchovecky/projects/coup_ppe/sims"
ARCH_DIR    = "/glade/derecho/scratch/bbuchovecky/archive"

LND_VARIABLES = ["TLAI", "TOTVEGC", "EFLX_LH_TOT", "FCTR", "FCEV", "FGEV", "TSA"]
ATM_VARIABLES = ["TREFHT", "PS", "PRECT", "TMQ", "FSNT", "FLNT", "CLDTOT"]

CLIM_YEAR_RANGE = ["1995-01", "1999-12"]
MAP_YEAR_RANGE  = ["1999-01", "1999-12"]

In [4]:
import xclimate as xclim

In [5]:
client_cluster = xclim.create_dask_cluster(
    account='UWAS0155',
    nworkers=2,
    ncores=1,
    nmem='5GB',
    walltime='02:00:00'
)

account:  UWAS0155
nworkers: 2
ncores:   1
nmemory:  5GB
walltime: 02:00:00
{}

To view the dask dashboard
Run the following command in your local terminal:
> ssh -N -L 8787:128.117.211.160:8787 bbuchovecky@casper.hpc.ucar.edu
Open the following link in your local browser:
> http://localhost:8787/status


In [53]:
# client_cluster[1]

In [52]:
xclim.close_dask_cluster(client_cluster)

In [ ]:
# --- Load data ---------------------------------------------------------------

ih0_lnd = load_variables(LND_VARIABLES, f"{CASE_PREFIX}.coupPPE.000", ARCH_DIR, domain="lnd")
ihx_lnd = load_variables(LND_VARIABLES, f"{CASE_PREFIX}.{CASE}", ARCH_DIR, domain="lnd")
fhx_lnd = load_variables(LND_VARIABLES, f"f.e21.FHIST_BGC.f19_f19_mg17.historical.{CASE}", ARCH_DIR, domain="lnd")

In [49]:
ih0_lnd

<xarray.Dataset> Size: 302MB
Dimensions:      (time: 780, lat: 96, lon: 144)
Coordinates:
  * time         (time) object 6kB 1950-01-01 00:00:00 ... 2014-12-01 00:00:00
  * lat          (lat) float32 384B -90.0 -88.11 -86.21 ... 86.21 88.11 90.0
  * lon          (lon) float32 576B 0.0 2.5 5.0 7.5 ... 350.0 352.5 355.0 357.5
Data variables:
    TLAI         (time, lat, lon) float32 43MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    TOTVEGC      (time, lat, lon) float32 43MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    EFLX_LH_TOT  (time, lat, lon) float32 43MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    FCTR         (time, lat, lon) float32 43MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    FCEV         (time, lat, lon) float32 43MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    FGEV         (time, lat, lon) float32 43MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
    TSA          (time, lat, lon) float32 43MB dask.array<chunksize=(1, 96, 144), meta=np.ndarray>
Attributes: (12/100)
    title:                                     CLM History file information
    comment:                                   NOTE: None of the variables ar...
    Conventions:                               CF-1.0
    history:                                   created on 04/05/26 13:09:31
    source:                                    Community Land Model CLM4.0
    hostname:                                  derecho
    ...                                        ...
    cft_irrigated_switchgrass:                 60
    cft_tropical_corn:                         61
    cft_irrigated_tropical_corn:               62
    cft_tropical_soybean:                      63
    cft_irrigated_tropical_soybean:            64
    time_period_freq:                          month_1

In [35]:
# --- Grid and conversion factors ---------------------------------------------

fh0 = glob(f"{ARCH_DIR}/{CASE_PREFIX}.coupPPE.000/lnd/hist/*.h0.*")
grid = xr.open_dataset(fh0[0], decode_timedelta=True, engine="netcdf4")[["area", "landfrac"]]

# la  : land area per grid cell [m²]; area is in km², landfrac is dimensionless
# lw  : fractional land area weight (sums to 1 globally)
la = (grid.area * 1e6 * grid.landfrac).fillna(0)
lw = la / la.sum()

ConversionFactor = namedtuple("ConversionFactor", ["cf", "cs", "unit", "kind"])
# cf : multiplicative conversion factor (lw for intensive, la/1e15 for PgC extensive)
# cs : additive constant (e.g., -273.15 K→°C for temperature variables)
cfs = {
    "TLAI":       ConversionFactor(lw,          0,       "m2/m2",    "intensive"),
    "TOTECOSYSC": ConversionFactor(la / 1e15,   0,       "PgC",      "extensive"),
    "TOTVEGC":    ConversionFactor(la / 1e15,   0,       "PgC",      "extensive"),
    "TOTSOMC":    ConversionFactor(la / 1e15,   0,       "PgC",      "extensive"),
    "RAIN":       ConversionFactor(lw,          0,       "mm/s",     "intensive"),
    "QRUNOFF":    ConversionFactor(lw,          0,       "mm/s",     "intensive"),
    "QSOIL":      ConversionFactor(lw,          0,       "mm/s",     "intensive"),
    "QVEGE":      ConversionFactor(lw,          0,       "mm/s",     "intensive"),
    "QVEGT":      ConversionFactor(lw,          0,       "mm/s",     "intensive"),
    "TWS":        ConversionFactor(lw,          0,       "mm",       "intensive"),
    "EFLX_LH_TOT":ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FCTR":       ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FCEV":       ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FGEV":       ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FSH":        ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FIRE":       ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FLDS":       ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FSR":        ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FSDS":       ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FGR":        ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "TSA":        ConversionFactor(lw,          -273.15, "degreeC",  "intensive"),
    "TREFHT":     ConversionFactor(lw,          -273.15, "degreeC",  "intensive"),
    "PS":         ConversionFactor(lw,          0,       "Pa",       "intensive"),
    "FSNT":       ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "FLNT":       ConversionFactor(lw,          0,       "W/m2",     "intensive"),
    "CLDTOT":     ConversionFactor(lw,          0,       "fraction", "intensive"),
    "PRECT":      ConversionFactor(lw*1000*86400, 0,     "mm/day",   "intensive"),
    "TMQ":        ConversionFactor(lw,          0,       "kg/m2",    "intensive"),
}

labels = {
    "intensive": "mean",
    "extensive": "total",
}

cmaps = {
    "veg": {
        "diff": "PRGn",
        "cont": "viridis",
        "vars": ["TLAI", "TOTVEGC"],
    },
    "water": {
        "diff": cmo.curl_r,
        "cont": cmo.rain,
        "vars": ["EFLX_LH_TOT", "FCTR", "FCEV", "FGEV", "PRECT", "TMQ"],
    },
    "temp": {
        "diff": "RdBu_r",
        "cont": "inferno",
        "vars": ["TSA", "TREFHT", "PS", "FSNT", "FLNT", "CLDTOT"],
    },
}

In [ ]:
# --- Run diagnostics ---------------------------------------------------------

plot_simple_diags(ihx_lnd, ih0_lnd, fhx_lnd, LND_VARIABLES, "lnd")
# plot_simple_diags(ihx_atm, ih0_atm, fhx_atm, ATM_VARIABLES, "atm")

In [ ]:
ihx = ihx_lnd
ih0 = ih0_lnd
fhx = fhx_lnd
variables = LND_VARIABLES
tag = "lnd"

#####

fig, axes = plt.subplots(
    nrows=len(variables),
    ncols=4,
    figsize=(20, 4 * len(variables)),
    layout="tight",
)
ax = axes.flatten()

for i, v in enumerate(variables):
    print(v)
    
    cm = cmaps["temp"]
    if v in cmaps["veg"]["vars"]:
        cm = cmaps["veg"]
    elif v in cmaps["water"]["vars"]:
        cm = cmaps["water"]
    elif v in cmaps["temp"]["vars"]:
        cm = cmaps["temp"]

    vmin = 0
    if v in ["TSA", "TREFHT"]:
        vmin = None

    smooth_window = 5

    # --- Time series (col 0) ---
    # (v + cs) * cf applies a shift (cs) and scaling (cf) to convert units;
    # .sum over (lat, lon) gives global integral or weighted mean depending on cf.
    ihx_ann = ((ihx[v] + cfs[v].cs) * cfs[v].cf).sum(dim=["lat", "lon"]).groupby("time.year").mean().chunk({'year':-1})
    ih0_ann = ((ih0[v] + cfs[v].cs) * cfs[v].cf).sum(dim=["lat", "lon"]).groupby("time.year").mean().chunk({'year':-1})
    fhx_ann = ((fhx[v] + cfs[v].cs) * cfs[v].cf).sum(dim=["lat", "lon"]).groupby("time.year").mean().chunk({'year':-1})

    ih0_ann.plot(ax=ax[4 * i], color="tab:orange", alpha=0.75, lw=0.75)
    ih0_ann.rolling(year=smooth_window, center=True).mean().plot(
        ax=ax[4 * i], color="tab:orange", alpha=1, lw=1.5, label="I coupPPE.000"
    )
    ihx_ann.plot(ax=ax[4 * i], color="tab:blue", alpha=0.75, lw=0.75)
    ihx_ann.rolling(year=smooth_window, center=True).mean().plot(
        ax=ax[4 * i], color="tab:blue", alpha=1, lw=1.5, label=f"I {CASE}"
    )
    fhx_ann.plot(ax=ax[4 * i], color="tab:red", alpha=0.75, lw=0.75)
    fhx_ann.rolling(year=smooth_window, center=True).mean().plot(
        ax=ax[4 * i], color="tab:green", alpha=1, lw=1.5, label=f"F {CASE}"
    )

    ax[4 * i].set_ylabel(f"{v} [{cfs[v].unit}]")
    ax[4 * i].set_title(f"global annual {labels[cfs[v].kind]} {v}")
    ax[4 * i].legend()

    # --- Seasonal cycle (col 1) ---
    ihx_clim = (
        (ihx[v] + cfs[v].cs) * cfs[v].cf
    ).sel(time=slice(CLIM_YEAR_RANGE[0], CLIM_YEAR_RANGE[1])).sum(dim=["lat", "lon"]).groupby("time.month").mean()
    ih0_clim = (
        (ih0[v] + cfs[v].cs) * cfs[v].cf
    ).sel(time=slice(CLIM_YEAR_RANGE[0], CLIM_YEAR_RANGE[1])).sum(dim=["lat", "lon"]).groupby("time.month").mean()
    fhx_clim = (
        (fhx[v] + cfs[v].cs) * cfs[v].cf
    ).sel(time=slice(CLIM_YEAR_RANGE[0], CLIM_YEAR_RANGE[1])).sum(dim=["lat", "lon"]).groupby("time.month").mean()

    ihx_clim.plot(ax=ax[4 * i + 1], color="tab:blue", label=f"I {CASE}")
    ih0_clim.plot(ax=ax[4 * i + 1], color="tab:orange", label="I coupPPE.000")
    fhx_clim.plot(ax=ax[4 * i + 1], color="tab:red", label=f"F {CASE}")

    ax[4 * i + 1].set_ylabel(f"{v} [{cfs[v].unit}]")
    ax[4 * i + 1].set_xlabel("month")
    ax[4 * i + 1].set_title(
        f"global clim {labels[cfs[v].kind]} {v} "
        f"{CLIM_YEAR_RANGE[0][:4]}-{CLIM_YEAR_RANGE[1][:4]}"
    )
    ax[4 * i + 1].legend()

    # --- Spatial map: ih0 climatology (col 2) ---
    (ih0[v] + cfs[v].cs).sel(
        time=slice(MAP_YEAR_RANGE[0], MAP_YEAR_RANGE[1])
    ).mean(dim="time").plot(
        ax=ax[4 * i + 2],
        vmin=vmin,
        cmap=cm["cont"],
        cbar_kwargs={"label": f"{v} [{cfs[v].unit}]"}
    )
    ax[4 * i + 2].set_title(f"{CASE} {v} {MAP_YEAR_RANGE[0][:4]}")

    # --- Spatial map: ihx - ih0 difference (col 3) ---
    (
        (ihx[v] + cfs[v].cs).sel(time=slice(MAP_YEAR_RANGE[0], MAP_YEAR_RANGE[1]))
        - (ih0[v] + cfs[v].cs).sel(time=slice(MAP_YEAR_RANGE[0], MAP_YEAR_RANGE[1]))
    ).mean(dim="time").plot(
        ax=ax[4 * i + 3],
        cmap=cm["diff"],
        norm=clr.CenteredNorm(),
        robust=True,
        cbar_kwargs={"label": f"{v} [{cfs[v].unit}]"}
    )
    ax[4 * i + 3].set_title(f"{CASE[-3:]}$-$000 {v} {MAP_YEAR_RANGE[0][:4]}")

    for j in range(2, 4):
        ax[4 * i + j].set_ylabel("")
        ax[4 * i + j].set_xlabel("")

In [ ]:
client_cluster[0].restart()